# Stage 2: Data Acquisition & Filtering
Capstone Project: Spam Message Analysis Using Malicious URLs

Keep only the needed columns, drop missing values, normalise text, keep the four valid classes, and remove empty URLs and zero-width/BOM characters.

In [1]:
from pathlib import Path

# Project root = parent of this notebook's stage folder (run the notebook from inside its own folder)
ROOT = Path.cwd().parent
RAW_CSV = ROOT / "dataset" / "dataset.csv"
INTER = ROOT / "dataset" / "intermediate"      # output of each stage becomes input of the next
RESULTS = ROOT / "results"
GRAPH_DIR = RESULTS / "graphs"
for p in (INTER, RESULTS, GRAPH_DIR):
    p.mkdir(parents=True, exist_ok=True)


In [2]:
import pandas as pd

try:
    df_raw = pd.read_csv(RAW_CSV)
except UnicodeDecodeError:
    df_raw = pd.read_csv(RAW_CSV, encoding="utf-8", encoding_errors="replace")
print("Raw shape:", df_raw.shape)


Raw shape: (651991, 2)


## Stage 2 — Data Acquisition & Filtering
Keep only the columns we need, drop rows with unusable values, and restrict to the four known, valid URL categories.

In [3]:
VALID_TYPES = {"benign", "phishing", "defacement", "malware"}

df = df_raw[["url", "type"]].copy()

# Drop missing values in either column
df = df.dropna(subset=["url", "type"]).copy()

# Normalize text
df["url"] = df["url"].astype(str).str.strip()
df["type"] = df["type"].astype(str).str.strip().str.lower()

# Filter out any unexpected/garbage category labels
before = len(df)
df = df[df["type"].isin(VALID_TYPES)].copy()
print(f"Rows removed for invalid/unknown type label: {before - len(df)}")

# Filter out empty-string URLs
before = len(df)
df = df[df["url"].str.len() > 0].copy()
print(f"Rows removed for empty URL: {before - len(df)}")

# Strip zero-width spaces / BOM characters that can silently corrupt string length & parsing
zero_width_pattern = r"[\u200b\ufeff]"
affected = df["url"].str.contains(zero_width_pattern, regex=True, na=False).sum()
df["url"] = df["url"].str.replace(zero_width_pattern, "", regex=True)
print(f"URLs cleaned of zero-width/BOM artifacts: {affected}")

print("\nShape after acquisition & filtering:", df.shape)
df["type"].value_counts()

Rows removed for invalid/unknown type label: 205
Rows removed for empty URL: 46


URLs cleaned of zero-width/BOM artifacts: 33

Shape after acquisition & filtering: (651066, 2)


type
benign        428016
defacement     96432
phishing       94098
malware        32520
Name: count, dtype: int64

In [4]:
df.to_csv(INTER / "stage2_filtered.csv", index=False)
print("Saved:", INTER / "stage2_filtered.csv")

Saved: C:\Users\pavan\Documents\Codex\2026-09-29\c-users-pavan-onedrive-documents-downloads\work\Spam-Message-Analysis-Using-Malicious-URLs\Spam-Message-Analysis-Using-Malicious-URLs\dataset\intermediate\stage2_filtered.csv
